# UnderCover: deciding claims with an LLM

A copy of the Week 1 Day 2 prototype, `w1d2.ipynb`, with every model at the `high` reasoning level for comparison. We load the UnderCover seed data and put each claim to a model through OpenRouter, with the customer's policy record and the policy wording. The model replies with a structured decision: approve, decline or refer, with a rationale and a payout. Then we score the answers against the handlers' decisions, compare three models and look at what OpenRouter charged.

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, Field

from sample_data import DATA, get_sample_data, load

## Connect to OpenRouter

The key comes from the `.env` file in this folder.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=300, max_retries=5)

## Load the seed data

`get_sample_data` returns a list of claims and the handlers' decisions on them, keyed by claim id. The claims, the policies and the policy wording are what the model sees. The decisions are the labels. They stay out of every prompt and are only used for scoring.

There are two sets of twenty-five claims. The default is the queue, which is the set we work against. `held_out=True` gives the held-out claims, for checking later that an improvement generalizes. `tiny=True` keeps five claims of either set, which fits inside a free model's daily allowance.

In [ ]:
TINY = False
HELD_OUT = False

claims, labels = get_sample_data(tiny=TINY, held_out=HELD_OUT)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text()

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

In [ ]:
claims[0]

## The structured output

A Pydantic class describes the reply we want. The rationale comes first so that the model reasons before it commits to a decision.

In [ ]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="Two or three sentences that explain the decision and name the section of the policy wording it rests on")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt holds the task and the whole policy wording, which is short enough to fit. The user prompt holds the policy record and the claim as JSON.

In [ ]:
SYSTEM_PROMPT = f"""You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the documents it asks for are attached. Work out the payout the way section 10 describes.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Give your rationale first, then the decision, then the payout.

{policy_wording}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and wording, then the policy record and the claim."""
    policy = policies[claim["policy_id"]]
    user_prompt = f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

## One call with a structured output

`parse` sends the Pydantic class as a JSON schema and hands back an instance of it. OpenRouter reports what each call cost in `usage.cost`, in dollars.

OpenRouter spreads an open model like gpt-oss-20b across a dozen providers, and they differ in precision and in how well they handle structured outputs. In testing, one provider returned an empty reply, one filled in the decision and left the rationale blank, and one rejected the request, although all three advertise structured outputs. So we name the providers we have tested. OpenRouter tries them in order and moves to the next when one is busy, which keeps the results repeatable from run to run.

All three models reason before they answer, and their default levels differ. This copy sets every model to `high`. Qwen 3.8 27B does not list `high` as a level, and OpenRouter accepts it and gives the model a reasoning budget between its `low` and its `xhigh`. The results record the reasoning tokens each call used.

In [ ]:
PROVIDERS = {
    "openai/gpt-oss-20b": ["deepinfra", "coreweave", "parasail"],
    "qwen/qwen3.8-27b": ["deepinfra", "akashml", "chutes"],
}
REASONING_EFFORT = "high"


def provider_settings(model):
    """Route only to providers that support structured outputs, and keep to the tested providers where we have named them."""
    settings = {"require_parameters": True}
    if model in PROVIDERS:
        settings["order"] = PROVIDERS[model]
        settings["allow_fallbacks"] = False
    return settings


def decide(model, claim):
    """Ask one model to decide one claim, and keep what OpenRouter says the call cost."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=REASONING_EFFORT,
        extra_body={"provider": provider_settings(model)},
    )
    answer = response.choices[0].message.parsed
    return {
        "claim_id": claim["claim_id"],
        "model": model,
        "provider": response.provider,
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "cost": response.usage.cost,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "reasoning_tokens": response.usage.completion_tokens_details.reasoning_tokens,
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
first = decide("openai/gpt-oss-20b", claims[0])
first

## Run every claim through each model

Every claim goes to each model, one call per claim. A small thread pool runs five at a time.

In [ ]:
MODELS = ["openai/gpt-oss-20b", "qwen/qwen3.8-27b", "openai/gpt-5.6-luna"]


def run(model):
    """Decide every claim with one model."""
    with ThreadPoolExecutor(max_workers=5) as pool:
        return list(pool.map(lambda claim: decide(model, claim), claims))


results = []
for model in MODELS:
    start = time.time()
    results += run(model)
    print(f"{model}: {len(claims)} claims in {time.time() - start:.0f} seconds")

## Score against the handlers

The decision is right when it matches the handler's. The payout is right when the decision is right and the amount is within a dollar of the handler's.

In [ ]:
scored = pd.DataFrame(results)
scored["label"] = scored["claim_id"].map(lambda claim_id: labels[claim_id]["decision"])
scored["label_payout"] = scored["claim_id"].map(lambda claim_id: labels[claim_id]["payout"])
scored["difficulty"] = scored["claim_id"].map(lambda claim_id: labels[claim_id]["difficulty"])
scored["decision_right"] = scored["decision"] == scored["label"]
scored["payout_right"] = scored["decision_right"] & ((scored["payout"] - scored["label_payout"]).abs() <= 1)

summary = scored.groupby("model").agg(
    decisions_right=("decision_right", "sum"),
    payouts_right=("payout_right", "sum"),
    claims=("claim_id", "count"),
    cost=("cost", "sum"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    reasoning_tokens=("reasoning_tokens", "sum"),
    mean_seconds=("seconds", "mean"),
)
summary["decision_accuracy"] = summary["decisions_right"] / summary["claims"]
summary["payout_accuracy"] = summary["payouts_right"] / summary["claims"]
summary

The same scores by the kind of claim. The policy wording settles the routine claims and the hard ones. The manual claims need the claims manual or the customer's history, and this prompt holds neither.

In [ ]:
scored.groupby(["model", "difficulty"]).agg(
    decisions_right=("decision_right", "sum"),
    payouts_right=("payout_right", "sum"),
    claims=("claim_id", "count"),
)

Which provider served each call. When the first provider on our list is busy, OpenRouter moves to the next.

In [ ]:
scored.groupby(["model", "provider"]).size()

## Side by side

One row per claim, with the handler's decision and each model's answer.

In [ ]:
scored["answer"] = scored["decision"] + " $" + scored["payout"].round(2).astype(str)
side_by_side = scored.pivot(index="claim_id", columns="model", values="answer")
side_by_side.insert(0, "handler", [f"{labels[claim_id]['decision']} ${labels[claim_id]['payout']}" for claim_id in side_by_side.index])
summaries = {claim["claim_id"]: claim["summary"] for claim in claims}
side_by_side.insert(0, "summary", [summaries[claim_id] for claim_id in side_by_side.index])
pd.set_option("display.max_colwidth", 70)
side_by_side

## Where the models got it wrong

Each miss with the model's rationale, beside the handler's reason and what the claim is in the set to test.

In [ ]:
misses = scored[~scored["payout_right"]].sort_values(["claim_id", "model"])

for miss in misses.to_dict("records"):
    label = labels[miss["claim_id"]]
    display(Markdown(f"""**{miss['claim_id']}, {miss['model']}**: {miss['decision']} ${miss['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {miss['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))

## What it cost

OpenRouter's reported cost for every call in this notebook, in dollars.

In [ ]:
total = scored["cost"].sum() + first["cost"]
print(f"Total for {len(scored) + 1} calls: ${total:.4f}")
scored.groupby("model")["cost"].agg(["sum", "mean", "max"])